# 00 — DYAMOND data inventory

Enumerate the Zarr stores under the Poseidon DYAMOND volume and record, for
downstream notebooks:

1. which store holds the **ocean surface fluxes** (`oceQnet`, `oceQsw`, `oceFWflx`, `oceTAUX/Y`);
2. whether the **GEOS atmosphere flux collections** (`EFLUX`, `HFLUX`, `SWGNT`, `LWGNT`) are
   present on SciServer (the Sci. Data descriptor lists 27 GEOS collections, but not all are
   necessarily mirrored to the ceph volume);
3. the **sign conventions** encoded in the variable attributes.

Reference: Menemenlis et al. (2026), *Sci. Data* — GEOS c1440 (~7 km) coupled to MITgcm
LLC2160 (~2–4 km), 2020-01-20 to 2021-03-26.

In [ ]:
# Environment check: this notebook must run on SciServer (Kraken domain,
# Oceanography image, "Poseidon DYAMOND (ceph)" data volume), or with
# DYAMOND_ROOT pointing at a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data volume is absent
print(f"DYAMOND root: {root}")

In [ ]:
from dyamond_fluxes import list_stores

stores = list_stores()
print(f"{len(stores)} zarr stores found:")
for s in stores:
    print("  ", s.relative_to(root))

In [ ]:
from dyamond_fluxes import find_stores_with

OCEAN_FLUX_VARS = ["oceQnet", "oceQsw", "oceFWflx", "oceSflux", "oceTAUX", "oceTAUY", "KPPhbl"]
GEOS_FLUX_VARS = ["EFLUX", "HFLUX", "SWGNT", "LWGNT", "LWGAB", "SWGDN", "PRECTOT"]

hits = find_stores_with(OCEAN_FLUX_VARS + GEOS_FLUX_VARS)
for store, variables in hits.items():
    print(f"{store.relative_to(root)}:\n    {variables}")

## Ocean surface flux store

Open the store containing `oceQnet` and inspect the flux variables. **Record the sign
convention from the attributes** — `fluxes.to_positive_down` relies on it, and the data
descriptor states fluxes as net *upward* while some MITgcm setups write positive-down.

In [ ]:
from dyamond_fluxes import open_store

ocean_store = next(s for s, v in hits.items() if "oceQnet" in v)
ds = open_store(ocean_store)
ds

In [ ]:
for name in ["oceQnet", "oceQsw", "oceFWflx", "oceTAUX", "oceTAUY"]:
    if name in ds:
        print(f"{name}: {dict(ds[name].attrs)}")

In [ ]:
# Grid and time layout (expected: 13 faces x 2160 x 2160, ~10k time steps).
print("dims:", dict(ds.sizes))
print("time range:", ds.time.values[0], "to", ds.time.values[-1])
grid_vars = [v for v in ["XC", "YC", "CS", "SN", "rA", "Depth"] if v in ds]
print("grid variables present:", grid_vars)

## Findings to carry forward

Fill in after running:

- Ocean flux store path: `…`
- `oceQnet` sign convention: `…`
- GEOS flux collections on SciServer: **yes/no** — if absent, notebook 02 falls back to the
  non-solar residual $Q_{net} - Q_{sw}$, and the full radiative/turbulent decomposition
  requires the GEOS collections from the NCCS Dataportal
  (https://gmao.gsfc.nasa.gov/global_mesoscale/dyamond_phaseII/data_access/).